In [2]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from ann_model_massspecgym import AnnRetrieval
from distance_functions import cosine_similarity, euclidean_distance
from chemembed_transforms import ChemEmbedSpecTransform, molformer_factory
from massspecgym.data.datasets import MassSpecDataset
from massspecgym.data.data_module import MassSpecDataModule
from massspecgym.data.transforms import InMemCachedMolTransform
# rdkit.Chem.Scaffolds.MurckoScaffold

# 1) MoLFormer Embedding

In [9]:
# MoLFormer (768 dims)
mol_transform_factory = molformer_factory
embedding_name = "molformer"
mol_embedding_dim = 768

selected_molecular_embedding = InMemCachedMolTransform(
    mol_transform_factory=mol_transform_factory,
    verbose=True
)

Loading weights:   0%|          | 0/207 [00:00<?, ?it/s]

[transformers] MolformerModel LOAD REPORT from: ibm/MoLFormer-XL-both-10pct
Key                                | Status     |  | 
-----------------------------------+------------+--+-
lm_head.transform.dense.weight     | UNEXPECTED |  | 
lm_head.transform.LayerNorm.weight | UNEXPECTED |  | 
lm_head.transform.dense.bias       | UNEXPECTED |  | 
lm_head.transform.LayerNorm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initializing InMemCachedMolTransform for MoLFormerMolTransform. Cache file not provided or missing; molecules will be transformed on the fly. Call `build_cache(smiles_list)` to create it.


## Calculate embeddings

In [10]:
dataset_path = Path("data/MSG_test_fold_w_chemont.tsv")
transformer = mol_transform_factory()

df = pd.read_csv(dataset_path, sep="\t")
df["molformer_embedding"] = df["smiles"].apply(transformer.from_smiles)
df.info()

Loading weights:   0%|          | 0/207 [00:00<?, ?it/s]

[transformers] MolformerModel LOAD REPORT from: ibm/MoLFormer-XL-both-10pct
Key                                | Status     |  | 
-----------------------------------+------------+--+-
lm_head.transform.dense.weight     | UNEXPECTED |  | 
lm_head.transform.LayerNorm.weight | UNEXPECTED |  | 
lm_head.transform.dense.bias       | UNEXPECTED |  | 
lm_head.transform.LayerNorm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14066 entries, 0 to 14065
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   identifier            14066 non-null  object 
 1   mzs                   14066 non-null  object 
 2   intensities           14066 non-null  object 
 3   smiles                14066 non-null  object 
 4   inchikey              14066 non-null  object 
 5   formula               14066 non-null  object 
 6   precursor_formula     14066 non-null  object 
 7   parent_mass           14066 non-null  float64
 8   precursor_mz          14066 non-null  float64
 9   adduct                14066 non-null  object 
 10  instrument_type       13816 non-null  object 
 11  collision_energy      9957 non-null   float64
 12  fold                  14066 non-null  object 
 13  simulation_challenge  14066 non-null  bool   
 14  chemont_tree          14056 non-null  object 
 15  molformer_embedding

In [ ]:
new_path = Path(str(dataset_path.with_suffix('')) + "_plus_embedding.tsv")
df.to_csv(new_path, sep="\t", index=False)

# 2) Load Dataset

In [18]:
test_dataset = MassSpecDataset(
    pth=dataset_path,
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
)

data_module_test = MassSpecDataModule(
    dataset=test_dataset,
    batch_size=1,
)

data_module_test.setup("test")

# 3) Model Evaluation

## Load model

In [ ]:
model_name = "ANN_trained_molformer.ckpt"
model = AnnRetrieval.load_from_checkpoint(
    f"models/{model_name}",
    mol_embedding_dim=mol_embedding_dim
)

with torch.no_grad():
    print(model.eval())

## Predict MoLFormer Embeddings

In [20]:
model_predictions = []
with torch.no_grad():
    for entry in data_module_test.test_dataloader():
        input = entry["spec"]
        if (torch.isnan(input).any()):
            input = torch.from_numpy(np.array([0] * mol_embedding_dim).astype(np.float32))
        input = input.to("cpu") 
        outputs = model(input).cpu()
        model_predictions.append(outputs)

pred_df = pd.concat([df, pd.DataFrame({"molformer_embedding_pred" : model_predictions}).reset_index(drop=True)], axis=1)
pred_df

,identifier,mzs,intensities,smiles,inchikey,formula,precursor_formula,parent_mass,precursor_mz,adduct,instrument_type,collision_energy,fold,simulation_challenge,chemont_tree,molformer_embedding,molformer_embedding_pred
0,MassSpecGymID0000203,"134.0964,234.1462,244.1305,262.1411","1.0,0.07907907907907907,0.3733733733733734,0.1...",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H58N3O9,783.408924,784.4162,[M+H]+,Orbitrap,35.0,test,True,"[0, 264, 1813, 1961, 1994]","[-0.089028426, 0.35098684, 0.5181631, 0.627438...","[[tensor(0.1692), tensor(0.2104), tensor(0.472..."
1,MassSpecGymID0000204,"91.0542,134.0964,244.1305","0.043043043043043044,1.0,0.04804804804804805",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H58N3O9,783.408924,784.4162,[M+H]+,Orbitrap,50.0,test,True,"[0, 264, 1813, 1961, 1994]","[-0.089028426, 0.35098684, 0.5181631, 0.627438...","[[tensor(0.2873), tensor(0.2440), tensor(0.520..."
2,MassSpecGymID0000205,"134.0964,216.1356,234.1462,244.1305,262.1411,3...","1.0,0.037037037037037035,0.15615615615615616,0...",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H58N3O9,783.408924,784.4162,[M+H]+,Orbitrap,30.0,test,True,"[0, 264, 1813, 1961, 1994]","[-0.089028426, 0.35098684, 0.5181631, 0.627438...","[[tensor(0.1106), tensor(0.2222), tensor(0.338..."
3,MassSpecGymID0000206,"134.0964,234.1462,244.1305,262.1411,362.1935,5...","0.2072072072072072,0.11311311311311312,1.0,0.6...",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H58N3O9,783.408924,784.4162,[M+H]+,Orbitrap,20.0,test,True,"[0, 264, 1813, 1961, 1994]","[-0.089028426, 0.35098684, 0.5181631, 0.627438...","[[tensor(0.2391), tensor(0.1781), tensor(0.467..."
4,MassSpecGymID0000208,"244.1332,262.1411,541.289,784.4168","0.5185185185185185,0.2722722722722723,0.085085...",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H58N3O9,783.408924,784.4162,[M+H]+,Orbitrap,10.0,test,True,"[0, 264, 1813, 1961, 1994]","[-0.089028426, 0.35098684, 0.5181631, 0.627438...","[[tensor(0.3669), tensor(0.3138), tensor(0.639..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14061,MassSpecGymID0400854,"55.055092,57.07082,58.024075,61.011219,62.9905...","0.0022286420583862614,0.01249259525147176,0.00...",C[Si]1(C)O[Si](C)(C)O[Si](C)(C)O[Si](C)(C)O[Si...,XMSXQFUHVRWGNA,C10H30O5Si5,C10H31O5Si5,370.094724,371.1020,[M+H]+,QTOF,NaN,test,False,"[0, 462, 1370, 3194, 4445]","[0.5263756, 0.28771174, 0.5084965, 0.73108816,...","[[tensor(0.5872), tensor(0.1763), tensor(0.783..."
14062,MassSpecGymID0401425,"53.188965,55.054928,56.187614,57.067844,57.070...","0.0014692404421424558,0.0075411048667320514,0....",Cc1cc(O)c(C(C)(C)C)cc1Sc1cc(C(C)(C)C)c(O)cc1C,HXIQYSLFEXIOAV,C22H30O2S,C22H31O2S,358.192724,359.2000,[M+H]+,QTOF,NaN,test,False,"[1, 423, 453, 555, 555]","[0.2832948, -0.026052035, 1.335323, 0.13672525...","[[tensor(0.2292), tensor(0.2192), tensor(1.022..."
14063,MassSpecGymID0402033,"135.043198,249.128204,263.141907,265.156799,28...","6.487956589565239e-05,3.5798763969405303e-06,3...",CCN1CC2(COC)CCC(OC)C34C5CC6C(OC)CC(OC(C)=O)(C5...,LYUPEIXJYAJCHL,C35H49NO9,C35H50NO9,627.342724,628.3500,[M+H]+,QTOF,NaN,test,False,"[0, 12, 259, 1551, 3788]","[0.4320165, -0.621472, 0.6958208, 0.28541386, ...","[[tensor(0.5787), tensor(-0.0268), tensor(0.47..."
14064,MassSpecGymID0402039,"102.090202,108.079903,154.121597,250.179199,25...","5.842000366504345e-06,1.7871895579441808e-06,0...",CCN1C[C@]2(COC)CC[C@H](O)[C@]34C1[C@@H]([C@H](...,PMCAUYATZKXGHU,C32H45NO8,C32H46NO8,571.317724,572.3250,[M+H]+,QTOF,NaN,test,False,"[0, 12, 259, 1551, 3788]","[0.65579975, -0.45199996, 0.75155145, 0.240650...","[[tensor(0.6486), tensor(-0.0649), tensor(0.65..."


In [28]:
new_path = Path(str(dataset_path.with_suffix('')) + "_plus_embedding_pred.tsv")
pred_df.to_csv(new_path, sep="\t", index=False)

## Compare with real embedding

In [25]:
ids = list(pred_df['identifier'])
pred_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14066 entries, 0 to 14065
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   identifier                14066 non-null  object 
 1   mzs                       14066 non-null  object 
 2   intensities               14066 non-null  object 
 3   smiles                    14066 non-null  object 
 4   inchikey                  14066 non-null  object 
 5   formula                   14066 non-null  object 
 6   precursor_formula         14066 non-null  object 
 7   parent_mass               14066 non-null  float64
 8   precursor_mz              14066 non-null  float64
 9   adduct                    14066 non-null  object 
 10  instrument_type           13816 non-null  object 
 11  collision_energy          9957 non-null   float64
 12  fold                      14066 non-null  object 
 13  simulation_challenge      14066 non-null  bool   
 14  chemon

In [26]:
results = list()

cosine = cosine_similarity(pred_df, "molformer_embedding", "molformer_embedding_pred")
results.append(pd.DataFrame({'identifier': ids, 'cosine_sim': cosine}))
euc = euclidean_distance(pred_df, "molformer_embedding", "molformer_embedding_pred")
results.append(pd.DataFrame({'identifier': ids, 'euclidean_dist': euc}))

print(len(cosine))
print(len(euc))
print(len(ids))

14066
14066
14066


In [27]:
measures = results[0]
for i in range(1, len(results)):
    measures = pd.merge(measures, results[i], on="identifier")

measures.info()
measures

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14066 entries, 0 to 14065
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   identifier      14066 non-null  object 
 1   cosine_sim      14066 non-null  float32
 2   euclidean_dist  14066 non-null  float32
dtypes: float32(2), object(1)
memory usage: 219.9+ KB


,identifier,cosine_sim,euclidean_dist
0,MassSpecGymID0000203,0.761149,10.036486
1,MassSpecGymID0000204,0.739842,10.529962
2,MassSpecGymID0000205,0.767589,9.961919
3,MassSpecGymID0000206,0.782684,9.603712
4,MassSpecGymID0000208,0.798874,9.288404
...,...,...,...
14061,MassSpecGymID0400854,0.653316,14.504745
14062,MassSpecGymID0401425,0.762149,11.733856
14063,MassSpecGymID0402033,0.757642,10.278358
14064,MassSpecGymID0402039,0.855012,8.529740


In [ ]:
measures.to_csv("data/similarity_measures.tsv", sep="\t", index=False)

: 

In [ ]:
dataset_path = Path("data/MSG_test_fold_w_chemont.tsv")
aaa = pd.read_csv(Path(str(dataset_path.with_suffix('')) + "_plus_embedding_pred.tsv"), sep="\t")
aaa.info()
aaa["molformer_embedding_pred"]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14066 entries, 0 to 14065
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   identifier                14066 non-null  object 
 1   mzs                       14066 non-null  object 
 2   intensities               14066 non-null  object 
 3   smiles                    14066 non-null  object 
 4   inchikey                  14066 non-null  object 
 5   formula                   14066 non-null  object 
 6   precursor_formula         14066 non-null  object 
 7   parent_mass               14066 non-null  float64
 8   precursor_mz              14066 non-null  float64
 9   adduct                    14066 non-null  object 
 10  instrument_type           13816 non-null  object 
 11  collision_energy          9957 non-null   float64
 12  fold                      14066 non-null  object 
 13  simulation_challenge      14066 non-null  bool   
 14  chemon